
# Part 1: Starting from High Explainability

Decision Trees and GAM's might be a natural starting place for models where explainability could be critical. Insight into exactly what is driving the classifier could drive your advertising spend in future campaigns or even as this one continues. 


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import DataLoader, TensorDataset
import time
from datetime import datetime
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score


In [ ]:


marketing_file = "./dataset/marketing_campaign.csv"

marketing_df = pd.read_csv("./dataset/marketing_campaign.csv", sep='\t', encoding="ascii")


In [ ]:
marketing_df.info()

In [ ]:
marketing_df.describe()

In [ ]:
# Fill null values with median
marketing_df['Income'] = marketing_df['Income'].fillna(marketing_df['Income'].median())

In [ ]:
# Dt_Customer ->datetime
marketing_df['Dt_Customer'] = pd.to_datetime(marketing_df['Dt_Customer'], format='%d-%m-%Y')

reference_date = datetime(2014, 7, 1)
marketing_df['Customer_Since_Months'] = (reference_date - marketing_df['Dt_Customer']).dt.days // 30

marketing_df.drop(columns=['Dt_Customer'], inplace=True)

In [ ]:
print(marketing_df['Education'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Education'], drop_first=False)

In [ ]:
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].replace({'Absurd': 'Other', 'YOLO': 'Other'})

print(marketing_df['Marital_Status'].value_counts())
marketing_df = pd.get_dummies(marketing_df, columns=['Marital_Status'], drop_first=False)

In [ ]:
marketing_df['Age'] = 2015 - marketing_df['Year_Birth']

marketing_df.drop(columns=['Year_Birth'], inplace=True)


In [ ]:
irrelevant_columns = ['ID', 'NumDealsPurchases', 'Response']
marketing_df = marketing_df.drop(columns=irrelevant_columns)

In [ ]:
marketing_df['AcceptedAny'] = (
    (marketing_df['AcceptedCmp1'] == 1) | 
    (marketing_df['AcceptedCmp2'] == 1) | 
    (marketing_df['AcceptedCmp3'] == 1) | 
    (marketing_df['AcceptedCmp4'] == 1) | 
    (marketing_df['AcceptedCmp5'] == 1)
).astype(int)

marketing_df.drop(columns=['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5'], inplace=True)

print(marketing_df[['AcceptedAny']].value_counts())

In [ ]:
marketing_df['Education'] = marketing_df[['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD']].idxmax(axis=1)

marketing_df['Marital_Status'] = marketing_df[['Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married', 
                                               'Marital_Status_Divorced', 'Marital_Status_Widow', 'Marital_Status_Alone',
                                               'Marital_Status_Other']].idxmax(axis=1)
marketing_df.drop(columns=['Education_Basic', 'Education_2n Cycle', 'Education_Graduation', 'Education_Master', 'Education_PhD',
                           'Marital_Status_Single', 'Marital_Status_Together', 'Marital_Status_Married', 'Marital_Status_Divorced', 
                           'Marital_Status_Widow', 'Marital_Status_Alone', 'Marital_Status_Other'], inplace=True)

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
marketing_df['Education'] = marketing_df['Education'].str.replace('Education_', '')
marketing_df['Marital_Status'] = marketing_df['Marital_Status'].str.replace('Marital_Status_', '')

print(marketing_df[['Education', 'Marital_Status']].head())

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
marketing_df['Education'] = label_enc.fit_transform(marketing_df['Education'])
marketing_df['Marital_Status'] = label_enc.fit_transform(marketing_df['Marital_Status'])

print(marketing_df[['Education', 'Marital_Status']].value_counts())

In [ ]:
def split_data(marketing_df, method='train_val_test', k=5, val_size=0.2, test_size=0.2):
    
    X = marketing_df.drop(columns=['AcceptedAny']).values
    y = marketing_df['AcceptedAny'].values

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    if method == 'train_val_test':
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        
        val_size_adjusted = val_size / (1 - test_size) 
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val, y_train_val, test_size=val_size_adjusted, random_state=42, stratify=y_train_val
        )

        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

        print(f"Train: {X_train_tensor.shape[0]}, Val: {X_val_tensor.shape[0]}, Test: {X_test_tensor.shape[0]}")
        return X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, X_test_tensor, y_test_tensor

    elif method == 'cross_val':
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
        folds = []

        for train_index, test_index in skf.split(X, y):
            X_train_fold, X_test_fold = X[train_index], X[test_index]
            y_train_fold, y_test_fold = y[train_index], y[test_index]

            X_train_tensor = torch.tensor(X_train_fold, dtype=torch.float32)
            y_train_tensor = torch.tensor(y_train_fold, dtype=torch.float32).unsqueeze(1)
            X_test_tensor = torch.tensor(X_test_fold, dtype=torch.float32)
            y_test_tensor = torch.tensor(y_test_fold, dtype=torch.float32).unsqueeze(1)

            folds.append((X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor))

        print(f"Train: {folds[0][0].shape[0]}, Test: {folds[0][2].shape[0]}")
        return folds


In [ ]:

# Drop constant columns that cause NaN when StandardScaler divides by std=0
# Z_CostContact is always 3, Z_Revenue is always 11 — they carry no information
constant_cols = [col for col in marketing_df.columns if marketing_df[col].nunique() <= 1]
print(f"Dropping constant columns: {constant_cols}")
marketing_df.drop(columns=constant_cols, inplace=True)

In [ ]:
# ============================================================
# Decision Tree
# ============================================================

from sklearn.tree import DecisionTreeClassifier

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Hyperparameter tuning: max_depth ---
max_depths = [5, 7, 10, 15, 20, None]
dt_results = []

for depth in max_depths:
    dt_model = DecisionTreeClassifier(max_depth=depth, random_state=42)

    start_time = time.time()
    dt_model.fit(X_train_scaled, y_train.ravel())
    end_time = time.time()
    training_time = end_time - start_time

    y_val_pred = dt_model.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred)

    dt_results.append({
        'max_depth': depth,
        'val_f1': val_f1,
        'training_time': training_time
    })

print("{:<12} {:<10} {:<15}".format("Max Depth", "F1-Score", "Training Time (s)"))
for r in dt_results:
    print("{:<12} {:<10.4f} {:<15.4f}".format(str(r['max_depth']), r['val_f1'], r['training_time']))

best_dt = max(dt_results, key=lambda x: x['val_f1'])
print(f"\nBest Max Depth: {best_dt['max_depth']}, Val F1: {best_dt['val_f1']:.4f}")

In [ ]:

# --- Evaluate best DT on test set ---
best_depth = best_dt['max_depth']
dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)

start_time = time.time()
dt_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()
dt_training_time = end_time - start_time

y_test_pred = dt_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"\nDecision Tree (max_depth={best_depth}) Test Results:")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Training Time: {dt_training_time:.4f} seconds")

In [ ]:

# --- Train size vs error ---
train_sizes = [400, 800, 1344]
dt_size_results = []

for size in train_sizes:
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
    dt_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = dt_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = dt_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    dt_size_results.append({'train_size': size, 'train_error': train_error, 'test_error': test_error, 'test_f1': test_f1})
    print(f"Train Size: {size} | Train Error: {train_error:.4f} | Test Error: {test_error:.4f} | Test F1: {test_f1:.4f}")

dt_size_results_arr = np.array([(r['train_size'], r['train_error'], r['test_error']) for r in dt_size_results])

plt.figure(figsize=(10, 5))
plt.title("Decision Tree: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(dt_size_results_arr[:, 0], dt_size_results_arr[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(dt_size_results_arr[:, 0], dt_size_results_arr[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("dt_train_size_vs_error.png")
plt.show()

In [ ]:
# --- Train size vs F1 ---
plt.figure(figsize=(10, 5))
plt.title("Decision Tree: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot([r['train_size'] for r in dt_size_results], [r['test_f1'] for r in dt_size_results], marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("dt_train_size_vs_f1.png")
plt.show()

In [ ]:
# ============================================================
# GAM (Generalized Additive Model)
# ============================================================

from pygam import LogisticGAM

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Hyperparameter tuning: n_splines and lam ---
gam_configs = [
    {'n_splines': n, 'lam': l}
    for n in [5, 10, 20]
    for l in [0.1, 1.0, 10.0]
]
gam_results = []

for cfg in gam_configs:
    gam_model = LogisticGAM(n_splines=cfg['n_splines'], lam=cfg['lam'])

    start_time = time.time()
    gam_model.fit(X_train_scaled, y_train.ravel())
    end_time = time.time()
    training_time = end_time - start_time

    y_val_pred = gam_model.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred)

    gam_results.append({
        **cfg,
        'val_f1': val_f1,
        'training_time': training_time
    })

print("{:<12} {:<10} {:<10} {:<15}".format("n_splines", "lam", "F1-Score", "Training Time (s)"))
for r in gam_results:
    print("{:<12} {:<10} {:<10.4f} {:<15.4f}".format(r['n_splines'], r['lam'], r['val_f1'], r['training_time']))

best_gam = max(gam_results, key=lambda x: x['val_f1'])
print(f"\nBest Config: n_splines={best_gam['n_splines']}, lam={best_gam['lam']}, Val F1: {best_gam['val_f1']:.4f}")

In [ ]:
# --- Evaluate best GAM on test set ---
gam_model = LogisticGAM(n_splines=best_gam['n_splines'], lam=best_gam['lam'])

start_time = time.time()
gam_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()
gam_training_time = end_time - start_time

y_test_pred = gam_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"\nGAM (n_splines={best_gam['n_splines']}, lam={best_gam['lam']}) Test Results:")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Training Time: {gam_training_time:.4f} seconds")

In [ ]:
# --- Train size vs error ---
train_sizes = [400, 800, 1344]
gam_size_results = []

for size in train_sizes:
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    gam_model = LogisticGAM(n_splines=best_gam['n_splines'], lam=best_gam['lam'])
    gam_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = gam_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = gam_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    gam_size_results.append({'train_size': size, 'train_error': train_error, 'test_error': test_error, 'test_f1': test_f1})
    print(f"Train Size: {size} | Train Error: {train_error:.4f} | Test Error: {test_error:.4f} | Test F1: {test_f1:.4f}")

gam_size_results_arr = np.array([(r['train_size'], r['train_error'], r['test_error']) for r in gam_size_results])

plt.figure(figsize=(10, 5))
plt.title("GAM: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(gam_size_results_arr[:, 0], gam_size_results_arr[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(gam_size_results_arr[:, 0], gam_size_results_arr[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("gam_train_size_vs_error.png")
plt.show()

In [ ]:
# --- Train size vs F1 ---
plt.figure(figsize=(10, 5))
plt.title("GAM: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot([r['train_size'] for r in gam_size_results], [r['test_f1'] for r in gam_size_results], marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("gam_train_size_vs_f1.png")
plt.show()

In [ ]:
# ============================================================
# XGBoost (Gradient Boosted Decision Tree)
# ============================================================

from xgboost import XGBClassifier

X_train, y_train, X_val, y_val, X_test, y_test = split_data(marketing_df, method='train_val_test')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --- Hyperparameter tuning ---
xgb_configs = [
    {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.1},
    {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1},
    {'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.1},
    {'n_estimators': 100, 'max_depth': 9, 'learning_rate': 0.1},
    {'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.1},
    {'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.1},
]

xgb_results = []

for cfg in xgb_configs:
    xgb_model = XGBClassifier(
        n_estimators=cfg['n_estimators'],
        max_depth=cfg['max_depth'],
        learning_rate=cfg['learning_rate'],
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )

    start_time = time.time()
    xgb_model.fit(X_train_scaled, y_train.ravel())
    end_time = time.time()
    training_time = end_time - start_time

    y_val_pred = xgb_model.predict(X_val_scaled)
    val_f1 = f1_score(y_val, y_val_pred)

    xgb_results.append({
        **cfg,
        'val_f1': val_f1,
        'training_time': training_time
    })

print("{:<15} {:<12} {:<15} {:<10} {:<15}".format(
    "n_estimators", "max_depth", "learning_rate", "F1-Score", "Time (s)"))
for r in xgb_results:
    print("{:<15} {:<12} {:<15} {:<10.4f} {:<15.4f}".format(
        r['n_estimators'], r['max_depth'], r['learning_rate'], r['val_f1'], r['training_time']))

best_xgb = max(xgb_results, key=lambda x: x['val_f1'])
print(f"\nBest Config: n_estimators={best_xgb['n_estimators']}, max_depth={best_xgb['max_depth']}, "
      f"lr={best_xgb['learning_rate']}, Val F1={best_xgb['val_f1']:.4f}")


In [ ]:

# --- Evaluate best XGBoost on test set ---
xgb_model = XGBClassifier(
    n_estimators=best_xgb['n_estimators'],
    max_depth=best_xgb['max_depth'],
    learning_rate=best_xgb['learning_rate'],
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

start_time = time.time()
xgb_model.fit(X_train_scaled, y_train.ravel())
end_time = time.time()
xgb_training_time = end_time - start_time

y_test_pred = xgb_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)

print(f"\nXGBoost Test Results:")
print(f"Accuracy: {accuracy:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"Training Time: {xgb_training_time:.4f} seconds")

In [ ]:

# --- Train size vs error ---
train_sizes = [400, 800, 1344]
xgb_size_results = []

for size in train_sizes:
    X_train_subset = X_train_scaled[:size]
    y_train_subset = y_train[:size]

    xgb_model = XGBClassifier(
        n_estimators=best_xgb['n_estimators'],
        max_depth=best_xgb['max_depth'],
        learning_rate=best_xgb['learning_rate'],
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    )
    xgb_model.fit(X_train_subset, y_train_subset.ravel())

    y_train_pred = xgb_model.predict(X_train_subset)
    train_error = 1 - accuracy_score(y_train_subset, y_train_pred)

    y_test_pred = xgb_model.predict(X_test_scaled)
    test_error = 1 - accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred)

    xgb_size_results.append({'train_size': size, 'train_error': train_error, 'test_error': test_error, 'test_f1': test_f1})
    print(f"Train Size: {size} | Train Error: {train_error:.4f} | Test Error: {test_error:.4f} | Test F1: {test_f1:.4f}")

xgb_size_results_arr = np.array([(r['train_size'], r['train_error'], r['test_error']) for r in xgb_size_results])

plt.figure(figsize=(10, 5))
plt.title("XGBoost: Train Size vs Training and Testing Error")
plt.xlabel("Training Size")
plt.ylabel("Error")
plt.plot(xgb_size_results_arr[:, 0], xgb_size_results_arr[:, 1], marker='o', linestyle='-', label="Training Error")
plt.plot(xgb_size_results_arr[:, 0], xgb_size_results_arr[:, 2], marker='s', linestyle='-', label="Testing Error")
plt.legend()
plt.grid(True)
plt.savefig("xgb_train_size_vs_error.png")
plt.show()

In [ ]:

# --- Train size vs F1 ---
plt.figure(figsize=(10, 5))
plt.title("XGBoost: Train Size vs Test F1-Score")
plt.xlabel("Training Size")
plt.ylabel("F1-Score")
plt.plot([r['train_size'] for r in xgb_size_results], [r['test_f1'] for r in xgb_size_results], marker='o', label="Test F1-Score")
plt.legend()
plt.grid(True)
plt.savefig("xgb_train_size_vs_f1.png")
plt.show()